# 1.3 Data Aggregation (tailored for demand prediction)

In this notebook we create a demand column in our dataset. This way we have our target variable which is essential for our data to be fully prepared for demand prediciton with ml models.

We have one big dataset with all trips now that we need to aggregate the data into different temporal resolutions, different spatial resolutions:

Temporal resolutions:
1. Hourly data
2. 2-hourly data
3. 6-hourly data
4. 24-hourly data

Spatial resolutions (h3):
1. medium resolution (res = 7)
2. low resolution (res = 6)

Note that we want to have both the hexagon codes and the community areas as data to compare performance differences in models between using the hexagon data and the community area data. We discard census tracts for our analysis since they are not available for most of the data.

### Data Requirements:

This notebook requires the following files: 

1. `01_05_trips_weather_merged.parquet` — local path: `data/merged/01_05_trips_weather_merged.parquet` — Sciebo path: `data_parquet/merged/01_05_trips_weather_merged.parquet`

Sample data is provided in the appropriate locations for testing purposes only. It will not produce meaningful results.

# Chapter 1: Feature Engineering on the Raw Dataset

All features that can be computed at the individual trip level are added to `trips_all` here, before any aggregation takes place. This ensures aggregated statistics (e.g. `avg_idle_time`) are computed from correct per-trip values rather than derived from already-summarised data.

In [33]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd
import numpy as np
from itertools import product

import h3
from shapely.geometry import Point, Polygon, shape
import geopandas as gpd
import holidays

# reset working dir
import os
from pathlib import Path

In [34]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Reset working directory                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import __main__
_nb = getattr(__main__, "__vsc_ipynb_file__", None) or os.environ.get("JPY_SESSION_NAME")
_start = Path(_nb).resolve().parent if _nb else Path.cwd()
os.chdir(next(p for p in [_start, *_start.parents] if (p / "pyproject.toml").exists()))
print(f"Working directory: {os.getcwd()}")

Working directory: C:\Users\Yannick Herrmann\Documents\Uni\AAA\AAA_Project\AAA_TA_2026


In [35]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Load data                               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

if Path("data/merged/01_05_trips_weather_merged.parquet").exists():
    trips_all = pd.read_parquet("data/merged/01_05_trips_weather_merged.parquet")
else:
    trips_all = pd.read_parquet("data/samples/SAMPLE_01_05_trips_weather_merged.parquet")

In [36]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Configurations                          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

START_TIME = pd.to_datetime("2025-01-01 00:00:00")
END_TIME   = pd.to_datetime("2026-01-01 00:00:00")

BUCKET_COL = "hour_stamp_since_epoch"

TEMPORAL_RESOLUTIONS = ["1h", "2h", "6h", "24h"]

SPATIAL_RESOLUTIONS = {
    "low":    "pickup_h3_res6",
    "medium": "pickup_h3_res7",
}

RESOLUTION_TO_H3 = {
    "low":    6,
    "medium": 7,
}

WEATHER_FEATURES = [
    "temperature_2m", "apparent_temperature", "precipitation",
    "snowfall", "snow_depth", "wind_speed_10m", "cloud_cover",
    "is_day", "rain", "sunshine_duration"
]

LEAKY_COLS = [
    "trip_id", "taxi_id",
    "trip_start_timestamp", "trip_end_timestamp",
    "trip_seconds", "trip_miles",
    "fare", "tips", "tolls", "extras", "trip_total",
    "payment_type", "company",
    "pickup_census_tract", "dropoff_census_tract",
    "pickup_community_area", "dropoff_community_area",
    "pickup_centroid_latitude", "pickup_centroid_longitude", "pickup_centroid_location",
    "dropoff_centroid_latitude", "dropoff_centroid_longitude", "dropoff_centroid_location",
    "dropoff_h3_low_resolution", "dropoff_h3_medium_resolution", "dropoff_h3_high_resolution",
    "start_year", "date",
    "relative_humidity_2m", "surface_pressure", "wind_speed_100m", "direct_radiation",
]


In [37]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Trip-level feature engineering          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# ── Area-type classification ──────────────────────────────────────────────────
# Bounding polygons for Chicago area types  (Shapely: (lon, lat) = (x, y))
_AREA_POLYGONS = {
    "downtown": [
        Polygon([(-87.648, 41.865), (-87.613, 41.865), (-87.613, 41.900), (-87.648, 41.900)]),
    ],
    "airport": [
        Polygon([(-87.945, 41.958), (-87.880, 41.958), (-87.880, 41.995), (-87.945, 41.995)]),
        Polygon([(-87.762, 41.773), (-87.724, 41.773), (-87.724, 41.803), (-87.762, 41.803)]),
    ],
}

def classify_latlon(lat, lon):
    point = Point(lon, lat)
    for area_type, polygons in _AREA_POLYGONS.items():
        if any(poly.contains(point) for poly in polygons):
            return area_type
    return "residential"

# ── Season / rush-hour lookup tables ─────────────────────────────────────────
_MONTH_TO_SEASON = {
    12: "winter",  1: "winter",  2: "winter",
     3: "spring",  4: "spring",  5: "spring",
     6: "summer",  7: "summer",  8: "summer",
     9: "autumn", 10: "autumn", 11: "autumn",
}

_RUSH_HOURS = {7, 8, 9, 16, 17, 18, 19}

# ── Compute features on every trip row ───────────────────────────────────────
trips_all["trip_start_timestamp"] = pd.to_datetime(trips_all["trip_start_timestamp"])
trips_all["trip_end_timestamp"]   = pd.to_datetime(trips_all["trip_end_timestamp"])

trips_all = trips_all.sort_values(["taxi_id", "trip_start_timestamp"]).reset_index(drop=True)

# idle time: gap in minutes between end of taxi's previous trip and start of this one
trips_all["idle_minutes"] = (
    (
        trips_all["trip_start_timestamp"]
        - trips_all.groupby("taxi_id")["trip_end_timestamp"].shift(1)
    ).dt.total_seconds() / 60
).clip(lower=0)

trips_all["season"]    = trips_all["trip_start_timestamp"].dt.month.map(_MONTH_TO_SEASON)
trips_all["rush_hour"] = trips_all["trip_start_timestamp"].dt.hour.isin(_RUSH_HOURS).astype(int)

# area_type via bounding-box lookup (rectangles, so bbox == polygon — no apply needed)
trips_all["area_type"] = "residential"
for _area_name, _polys in _AREA_POLYGONS.items():
    for _poly in _polys:
        _minx, _miny, _maxx, _maxy = _poly.bounds
        _mask = (
            trips_all["pickup_centroid_longitude"].between(_minx, _maxx)
            & trips_all["pickup_centroid_latitude"].between(_miny, _maxy)
        )
        trips_all.loc[_mask, "area_type"] = _area_name

print(f"trips_all shape : {trips_all.shape}")
print(f"idle_minutes NaN (first trip per taxi) : {trips_all['idle_minutes'].isna().sum()}")
print(f"area_type distribution:\n{trips_all['area_type'].value_counts()}")
print(f"season distribution  :\n{trips_all['season'].value_counts()}")


trips_all shape : (10835, 43)
idle_minutes NaN (first trip per taxi) : 2634
area_type distribution:
area_type
downtown       4862
residential    3962
airport        2011
Name: count, dtype: int64
season distribution  :
season
summer    2987
spring    2828
autumn    2733
winter    2287
Name: count, dtype: int64


In [38]:
print(trips_all.columns)
trips_all.head()

Index(['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp',
       'trip_seconds', 'trip_miles', 'pickup_community_area',
       'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude',
       'pickup_centroid_longitude', 'dropoff_centroid_latitude',
       'dropoff_centroid_longitude', 'pickup_h3_res6', 'dropoff_h3_res6',
       'pickup_h3_res7', 'dropoff_h3_res7', 'trip_start_hour', 'index',
       'temperature_2m', 'relative_humidity_2m', 'apparent_temperature',
       'precipitation', 'rain', 'snowfall', 'snow_depth', 'surface_pressure',
       'cloud_cover', 'wind_speed_10m', 'wind_speed_100m', 'is_day',
       'sunshine_duration', 'direct_radiation', 'idle_minutes', 'season',
       'rush_hour', 'area_type'],
      dtype='str')


,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area,dropoff_community_area,fare,tips,...,cloud_cover,wind_speed_10m,wind_speed_100m,is_day,sunshine_duration,direct_radiation,idle_minutes,season,rush_hour,area_type
0,883fbb20c5c26040a96646708940260735d8e4f8,0044e6c0d091476299b99345501f756b23632a96cbaf40...,2025-08-06 07:45:00,2025-08-06 08:00:00,661.0,2.08,8,24,9.50,2.91,...,79.0,3.960000,7.689603,1.0,3600.0,33.0,NaN,summer,1,downtown
1,4709d92874e8607bc5e3f691ff4d7d5b3705bf5a,0044e6c0d091476299b99345501f756b23632a96cbaf40...,2025-10-03 11:00:00,2025-10-03 11:15:00,578.0,1.34,8,32,8.50,1.20,...,0.0,4.846648,7.704336,1.0,3600.0,440.0,83700.0,autumn,0,downtown
2,65224cf374d8f330364da1eb3f3443e6fd1f7f7b,005a35095f423d97a46c9015d6602e614addb59fe659b2...,2025-03-20 16:30:00,2025-03-20 16:30:00,277.0,0.38,32,32,4.75,3.00,...,2.0,12.738099,18.710478,1.0,3600.0,498.0,NaN,spring,1,downtown
3,3a2fd3f97fba528688fba150f833eb7842a0bc19,005a35095f423d97a46c9015d6602e614addb59fe659b2...,2025-07-01 09:15:00,2025-07-01 09:30:00,569.0,1.15,28,8,7.25,0.00,...,0.0,6.854195,9.913082,1.0,3600.0,385.0,147885.0,summer,1,residential
4,7252525c9439b6d899cd9c6024ae64a9c72244d3,005a35095f423d97a46c9015d6602e614addb59fe659b2...,2025-08-01 20:00:00,2025-08-01 20:15:00,684.0,1.47,32,28,8.25,3.00,...,0.0,11.212135,21.031521,1.0,3600.0,36.0,45270.0,summer,0,downtown


# Chapter 2: Aggregation and Dataset Construction

The enriched `trips_all` is aggregated into every combination of temporal and spatial resolution required for model training. The pipeline then completes the grid (zero-demand cells), fills weather, and adds bucket-level features that are constant within a time bucket.

## 2.1 Aggregate enriched trips

Group by `(time_bucket, spatial_unit)` and compute:
- `trip_count` — number of trips
- `active_taxis` — unique taxis operating
- `avg_idle_time` — mean of per-trip `idle_minutes`
- Mean of all numeric trip and weather columns

In [39]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Aggregate data — Hexagons               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# idle_minutes is already in trips_all from Chapter 1 and will be averaged to avg_idle_time
numeric_cols = trips_all.select_dtypes(include="number").columns.tolist()

for freq in TEMPORAL_RESOLUTIONS:
    trips_all["time_bucket"] = trips_all["trip_start_timestamp"].dt.floor(freq)

    for spat_res, hex in SPATIAL_RESOLUTIONS.items():

        trip_count = (
            trips_all
            .groupby(["time_bucket", hex])
            .size()
            .rename("trip_count")
        )

        active_taxis = (
            trips_all
            .groupby(["time_bucket", hex])["taxi_id"]
            .nunique()
            .rename("active_taxis")
        )

        agg = (
            trips_all
            .groupby(["time_bucket", hex])[numeric_cols]
            .mean()
        )

        # compute idle time of taxis
        agg = agg.join(trip_count).join(active_taxis).reset_index()
        agg = agg.rename(columns={"idle_minutes": "avg_idle_time"})

        out = f"data/aggregated/hexagon/trip_count_agg/hex_agg_trips_{freq}_{f"{spat_res}_res"}.parquet"
        
        # create hexagon folder in data/aggregated if it doesnt exist yet
        os.makedirs("data/aggregated/hexagon/trip_count_agg", exist_ok=True)
        
        agg.to_parquet(out, index=False)
        print(f"[{freq} | {f"{spat_res}"}]  shape={agg.shape}  saved → {out}")

[1h | low]  shape=(9219, 34)  saved → data/aggregated/hexagon/trip_count_agg/hex_agg_trips_1h_low_res.parquet
[1h | medium]  shape=(10252, 34)  saved → data/aggregated/hexagon/trip_count_agg/hex_agg_trips_1h_medium_res.parquet
[2h | low]  shape=(8210, 34)  saved → data/aggregated/hexagon/trip_count_agg/hex_agg_trips_2h_low_res.parquet
[2h | medium]  shape=(9733, 34)  saved → data/aggregated/hexagon/trip_count_agg/hex_agg_trips_2h_medium_res.parquet
[6h | low]  shape=(6093, 34)  saved → data/aggregated/hexagon/trip_count_agg/hex_agg_trips_6h_low_res.parquet
[6h | medium]  shape=(8490, 34)  saved → data/aggregated/hexagon/trip_count_agg/hex_agg_trips_6h_medium_res.parquet
[24h | low]  shape=(3619, 34)  saved → data/aggregated/hexagon/trip_count_agg/hex_agg_trips_24h_low_res.parquet
[24h | medium]  shape=(6424, 34)  saved → data/aggregated/hexagon/trip_count_agg/hex_agg_trips_24h_medium_res.parquet


In [40]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Aggregate data — Community Areas        #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# We discard census tracts here: they are unavailable for most of the data, so
# we aggregate on the community-area level instead.
COMMUNITY_AREA_COL = "pickup_community_area"

numeric_cols_ca = [c for c in numeric_cols if c != COMMUNITY_AREA_COL]

for freq in TEMPORAL_RESOLUTIONS:
    trips_all["time_bucket"] = trips_all["trip_start_timestamp"].dt.floor(freq)

    trip_count = (
        trips_all
        .groupby(["time_bucket", COMMUNITY_AREA_COL])
        .size()
        .rename("trip_count")
    )

    active_taxis = (
        trips_all
        .groupby(["time_bucket", COMMUNITY_AREA_COL])["taxi_id"]
        .nunique()
        .rename("active_taxis")
    )

    agg = (
        trips_all
        .groupby(["time_bucket", COMMUNITY_AREA_COL])[numeric_cols_ca]
        .mean()
    )

    agg = agg.join(trip_count).join(active_taxis).reset_index()
    agg = agg.rename(columns={"idle_minutes": "avg_idle_time"})

    out = f"data/aggregated/community_area/trip_count_agg/community_area_agg_trips_{freq}_.parquet"
    os.makedirs(os.path.dirname(out), exist_ok=True)
    agg.to_parquet(out, index=False)
    print(f"[{freq} | community_area]  shape={agg.shape}  saved → {out}")

[1h | community_area]  shape=(9501, 33)  saved → data/aggregated/community_area/trip_count_agg/community_area_agg_trips_1h_.parquet
[2h | community_area]  shape=(8537, 33)  saved → data/aggregated/community_area/trip_count_agg/community_area_agg_trips_2h_.parquet
[6h | community_area]  shape=(6466, 33)  saved → data/aggregated/community_area/trip_count_agg/community_area_agg_trips_6h_.parquet
[24h | community_area]  shape=(3972, 33)  saved → data/aggregated/community_area/trip_count_agg/community_area_agg_trips_24h_.parquet


## 2.2 Helper functions

Functions for grid completion, weather filling, encoding, and bucket-level feature engineering. These operate on the aggregated data and handle zero-demand rows that have no corresponding trips.

In [41]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Fill non-demand hours                   #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

def fill_missing_hours(df, time_col="time_bucket", hex_col="h3_hexagon_low", start_time=START_TIME, end_time=END_TIME):
    df[time_col] = pd.to_datetime(df[time_col])

    expected_hours = pd.date_range(start=start_time, end=end_time, freq="h")
    all_hexagons = df[hex_col].unique()

    # full cartesian product of all hexagons x all hours
    full_index = pd.MultiIndex.from_product(
        [all_hexagons, expected_hours],
        names=[hex_col, time_col]
    )

    df_full = full_index.to_frame(index=False)
    df = df_full.merge(df, on=[hex_col, time_col], how="left")

    # compute after merge so every row (including newly added empty-hour rows) has it
    df["hour_stamp_since_epoch"] = (
        df[time_col].astype("int64") // 10**6 // 3600
    )

    n_missing = len(df_full) - len(df.dropna(subset=[c for c in df.columns if c not in [hex_col, time_col, "hour_stamp_since_epoch"]]))
    print(f"Total expected rows : {len(df_full)}")
    print(f"Missing combinations: {n_missing}")

    df = df.sort_values([hex_col, time_col]).reset_index(drop=True)

    print(f"Done. New shape: {df.shape}")
    return df

In [42]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Non-demand hexagons                     #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

CITY_POLYGONS = {
    "chicago": {
        "type": "Polygon",
        "coordinates": [[
            [-87.94011, 41.64454],
            [-87.52414, 41.64454],
            [-87.52414, 42.02304],
            [-87.94011, 42.02304],
            [-87.94011, 41.64454],
        ]]
    },
}


def get_city_hexagons(city, resolution):
    """
    Get all H3 hexagons covering a city at a given resolution.

    Args:
        city:       City name as a lowercase string (e.g. "chicago").
        resolution: H3 resolution integer (e.g. 7, 8, 9).
    """
    # if city.lower() not in CITY_POLYGONS:
    #     raise ValueError(f"City '{city}' not found. Available cities: {list(CITY_POLYGONS.keys())}")

    polygon = CITY_POLYGONS[city.lower()]
    hexagons = list(h3.geo_to_cells(polygon, resolution))
    print(f"  Found {len(hexagons)} hexagons for {city} at resolution {resolution}")
    return hexagons


def add_non_demand_hex(df, hex_col, bucket_col, city, resolution):
    """
    Ensure every (time_bucket, hexagon) combination exists in the DataFrame,
    filling missing rows with trip_count=0 for hexagons with no demand.

    Args:
        df:         Pre-aggregated DataFrame with trip_count and hex/bucket columns.
        hex_col:    Column name for H3 hex cell identifier.
        bucket_col: Column name for time bucket.
        city:       City name as a lowercase string (e.g. "chicago").
        resolution: H3 resolution integer (e.g. 7, 8, 9).
    """

    city_hexagons = get_city_hexagons(city, resolution)
    all_buckets   = sorted(df[bucket_col].unique())

    full_grid = pd.DataFrame(
        list(product(all_buckets, city_hexagons)),
        columns=[bucket_col, hex_col]
    )

    df = full_grid.merge(df, on=[bucket_col, hex_col], how="left")
    df["trip_count"] = df["trip_count"].fillna(0).astype(int)

    # Forward fill numeric context columns (e.g. weather) per hexagon
    numeric_context = df.select_dtypes(include="number").columns.difference(["trip_count"])
    df[numeric_context] = df.groupby(hex_col)[numeric_context].ffill()

    # print(f"  Grid shape after fill: {df.shape}")
    return df

In [43]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Non-demand community areas              #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

def add_non_demand_ca(df, ca_col, bucket_col):
    """
    Ensure every (time_bucket, community_area) combination exists,
    filling missing rows with trip_count=0.

    Args:
        df:         Pre-aggregated DataFrame with trip_count and ca/bucket columns.
        ca_col:     Column name for community area identifier.
        bucket_col: Column name for time bucket.
    """
    all_cas     = df[ca_col].dropna().unique()
    all_buckets = sorted(df[bucket_col].dropna().unique())

    full_grid = pd.DataFrame(
        list(product(all_buckets, all_cas)),
        columns=[bucket_col, ca_col]
    )

    df = full_grid.merge(df, on=[bucket_col, ca_col], how="left")
    df["trip_count"] = df["trip_count"].fillna(0).astype(int)

    # Forward fill numeric context columns (e.g. weather) per community area
    numeric_context = df.select_dtypes(include="number").columns.difference(["trip_count"])
    df[numeric_context] = df.groupby(ca_col)[numeric_context].ffill()

    return df

In [44]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# NaN Filling (weather, time_bucket)      #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

chicago_holidays = holidays.country_holidays("US", subdiv="IL")

def fill_time_bucket(df):
    """
    Exctract missing temporal data from hour_stam_since_epoch.
    """
    mask = df["time_bucket"].isna()
    df.loc[mask, "time_bucket"] = pd.to_datetime(
        df.loc[mask, "hour_stamp_since_epoch"] * 3600, unit="s"
    )

    df["start_month"] = df["time_bucket"].dt.month
    df["start_hour"]  = df["time_bucket"].dt.hour
    df["day_of_week"] = df["time_bucket"].dt.dayofweek
    df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)
    df["is_rush_hour"] = df["start_hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)
    df["is_holiday"]  = df["time_bucket"].dt.date.map(lambda d: int(d in chicago_holidays))


    print(f"  Filled {mask.sum()} missing time_bucket values")
    return df


def fill_weather(df, bucket_col, weather_cols):
    """
    Fill missing weather data based on time_bucket, assuming city-wide uniform weather.

    Args:
        df:          DataFrame with weather columns and time_bucket.
        bucket_col:  Column name for time bucket.
        weather_cols: List of weather column names to fill.
    """
    weather_lookup = (
        df.dropna(subset=weather_cols)
        .groupby(bucket_col)[weather_cols]
        .first()
    )

    df = df.drop(columns=weather_cols).merge(weather_lookup, on=bucket_col, how="left")

    still_missing = df[weather_cols].isna().sum()
    
    # print(weather_lookup)

    print(f"  Weather NaNs remaining after fill:\n{still_missing}")
    return df

In [45]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Sine/Cosine Transformation              #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

def sine_cosine_encode(df, col, period):
    """
    :param df: DataFrame containing the column to encode.
    :param col: Name of the column to encode (e.g. "hour", "month").
    :param period: The period of the cycle (e.g. 24 for hours, 12 for months).
    """
    df[f'{col}_sin'] = np.sin(2 * np.pi * df[col] / period)
    df[f'{col}_cos'] = np.cos(2 * np.pi * df[col] / period)
    return df

In [46]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Trip characteristics                    #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

def add_trip_characteristics(df):
    df["avg_trip_duration"] = df["trip_seconds"]
    df["avg_trip_distance"] = df["trip_miles"]
    df["avg_fare"]          = df["fare"]
    df["avg_trip_total"]    = df["trip_total"]
    df["avg_tip"]           = df["tips"]
    df["tip_rate"]          = np.where(df["fare"] > 0, df["tips"] / df["fare"], np.nan)
    return df

## 2.3 Build final datasets

Load each aggregated parquet file, complete the spatio-temporal grid, and apply bucket-level features.

In [47]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Create datasets                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

datasets = {}

for temp_res in TEMPORAL_RESOLUTIONS:
    for spat_res, hex_col in SPATIAL_RESOLUTIONS.items():

        resolution = RESOLUTION_TO_H3[spat_res]

        agg_df = pd.read_parquet(f"data/aggregated/hexagon/trip_count_agg/hex_agg_trips_{temp_res}_{spat_res}_res.parquet")
        print(f"\nBuilding [{temp_res} | {spat_res}] resolution using {hex_col}...")

        # 1. Grid & completeness
        agg_df = fill_missing_hours(agg_df, time_col="time_bucket", hex_col=hex_col, start_time=START_TIME, end_time=END_TIME)
        agg_df = add_non_demand_hex(agg_df, hex_col, BUCKET_COL, city="chicago", resolution=resolution)
        agg_df = fill_time_bucket(agg_df)
        agg_df = fill_weather(agg_df, bucket_col=BUCKET_COL, weather_cols=WEATHER_FEATURES)

        # 2. Temporal features
        agg_df = sine_cosine_encode(agg_df, "start_hour", 24)
        agg_df = sine_cosine_encode(agg_df, "start_month", 12)
        agg_df = sine_cosine_encode(agg_df, "day_of_week", 7)

        # 3. Demand features
        agg_df = add_trip_characteristics(agg_df)
        # 4. Fill zero-demand rows with 0
        agg_df = agg_df.fillna(0)

        # 5. Drop cross-spatial columns
        agg_df = agg_df.drop(columns=['pickup_census_tract', 'dropoff_census_tract', 'pickup_community_area', 'dropoff_community_area'], errors="ignore")

        datasets[f"{temp_res}_{spat_res}"] = agg_df


Building [1h | low] resolution using pickup_h3_res6...
Total expected rows : 254069
Missing combinations: 246969
Done. New shape: (254069, 35)
  Found 41 hexagons for chicago at resolution 6
  Filled 148937 missing time_bucket values
  Weather NaNs remaining after fill:
temperature_2m          0
apparent_temperature    0
precipitation           0
snowfall                0
snow_depth              0
wind_speed_10m          0
cloud_cover             0
is_day                  0
rain                    0
sunshine_duration       0
dtype: int64

Building [1h | medium] resolution using pickup_h3_res7...
Total expected rows : 1279106
Missing combinations: 1271292
Done. New shape: (1279106, 35)
  Found 279 hexagons for chicago at resolution 7
  Filled 1209018 missing time_bucket values
  Weather NaNs remaining after fill:
temperature_2m          0
apparent_temperature    0
precipitation           0
snowfall                0
snow_depth              0
wind_speed_10m          0
cloud_cover        

In [48]:
test = datasets["1h_low"]
test.isnull().sum()[test.isnull().sum() > 0]

Series([], dtype: int64)

In [49]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Create datasets — Community Areas       #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

datasets_ca = {}

for temp_res in TEMPORAL_RESOLUTIONS:
    print(f"\nBuilding [{temp_res} | community_area]...")

    agg_df = pd.read_parquet(f"data/aggregated/community_area/trip_count_agg/community_area_agg_trips_{temp_res}_.parquet")

    # 1. Grid & completeness
    agg_df = fill_missing_hours(agg_df, time_col="time_bucket", hex_col=COMMUNITY_AREA_COL, start_time=START_TIME, end_time=END_TIME)
    agg_df = add_non_demand_ca(agg_df, COMMUNITY_AREA_COL, BUCKET_COL)
    agg_df = fill_time_bucket(agg_df)
    agg_df = fill_weather(agg_df, bucket_col=BUCKET_COL, weather_cols=WEATHER_FEATURES)

    # 2. Temporal features
    agg_df = sine_cosine_encode(agg_df, "start_hour", 24)
    agg_df = sine_cosine_encode(agg_df, "start_month", 12)
    agg_df = sine_cosine_encode(agg_df, "day_of_week", 7)

    # 3. Demand features
    agg_df = add_trip_characteristics(agg_df)
    # 4. Fill zero-demand rows with 0
    agg_df = agg_df.fillna(0)

    # 5. Drop cross-spatial columns
    agg_df = agg_df.drop(columns=['pickup_census_tract', 'dropoff_census_tract', 'dropoff_community_area', 'pickup_h3_low_resolution', 'pickup_h3_medium_resolution', 'pickup_h3_high_resolution', 'dropoff_h3_low_resolution', 'dropoff_h3_medium_resolution', 'dropoff_h3_high_resolution'], errors="ignore")



    datasets_ca[temp_res] = agg_df


Building [1h | community_area]...
Total expected rows : 657075
Missing combinations: 649736
Done. New shape: (657075, 34)
  Filled 0 missing time_bucket values
  Weather NaNs remaining after fill:
temperature_2m          0
apparent_temperature    0
precipitation           0
snowfall                0
snow_depth              0
wind_speed_10m          0
cloud_cover             0
is_day                  0
rain                    0
sunshine_duration       0
dtype: int64

Building [2h | community_area]...
Total expected rows : 657075
Missing combinations: 650388
Done. New shape: (657075, 34)
  Filled 0 missing time_bucket values
  Weather NaNs remaining after fill:
temperature_2m          0
apparent_temperature    0
precipitation           0
snowfall                0
snow_depth              0
wind_speed_10m          0
cloud_cover             0
is_day                  0
rain                    0
sunshine_duration       0
dtype: int64

Building [6h | community_area]...
Total expected rows : 6

## 2.4 Save

In [50]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save datasets - Hexagons                #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

for key, df in datasets.items():
    out = f"data/aggregated/hexagon/demand_hex_{key}.parquet"
    df.to_parquet(out, index=False)
    print(f"Saved → {out}")

Saved → data/aggregated/hexagon/demand_hex_1h_low.parquet
Saved → data/aggregated/hexagon/demand_hex_1h_medium.parquet
Saved → data/aggregated/hexagon/demand_hex_2h_low.parquet
Saved → data/aggregated/hexagon/demand_hex_2h_medium.parquet
Saved → data/aggregated/hexagon/demand_hex_6h_low.parquet
Saved → data/aggregated/hexagon/demand_hex_6h_medium.parquet
Saved → data/aggregated/hexagon/demand_hex_24h_low.parquet
Saved → data/aggregated/hexagon/demand_hex_24h_medium.parquet


In [51]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save datasets — Community Areas         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

for key, df in datasets_ca.items():
    out = f"data/aggregated/community_area/demand_ca_{key}.parquet"
    os.makedirs(os.path.dirname(out), exist_ok=True)
    df.to_parquet(out, index=False)
    print(f"Saved → {out}")

Saved → data/aggregated/community_area/demand_ca_1h.parquet
Saved → data/aggregated/community_area/demand_ca_2h.parquet
Saved → data/aggregated/community_area/demand_ca_6h.parquet
Saved → data/aggregated/community_area/demand_ca_24h.parquet


In [52]:
test = datasets["1h_low"]

In [53]:
test.columns

Index(['hour_stamp_since_epoch', 'pickup_h3_res6', 'time_bucket',
       'trip_seconds', 'trip_miles', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'pickup_centroid_latitude', 'pickup_centroid_longitude',
       'dropoff_centroid_latitude', 'dropoff_centroid_longitude', 'index',
       'relative_humidity_2m', 'surface_pressure', 'wind_speed_100m',
       'direct_radiation', 'avg_idle_time', 'rush_hour', 'trip_count',
       'active_taxis', 'start_month', 'start_hour', 'day_of_week',
       'is_weekend', 'is_rush_hour', 'is_holiday', 'temperature_2m',
       'apparent_temperature', 'precipitation', 'snowfall', 'snow_depth',
       'wind_speed_10m', 'cloud_cover', 'is_day', 'rain', 'sunshine_duration',
       'start_hour_sin', 'start_hour_cos', 'start_month_sin',
       'start_month_cos', 'day_of_week_sin', 'day_of_week_cos',
       'avg_trip_duration', 'avg_trip_distance', 'avg_fare', 'avg_trip_total',
       'avg_tip', 'tip_rate'],
      dtype='str')

In [54]:
test.head()

,hour_stamp_since_epoch,pickup_h3_res6,time_bucket,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total,...,start_month_sin,start_month_cos,day_of_week_sin,day_of_week_cos,avg_trip_duration,avg_trip_distance,avg_fare,avg_trip_total,avg_tip,tip_rate
0,482136,86266452fffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
1,482136,862664c37ffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
2,482136,862664ce7ffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
3,482136,862664197ffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
4,482136,862664557ffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0


In [55]:
test = pd.read_parquet("data/aggregated/hexagon/demand_hex_1h_low.parquet")
print(len(test))
print(test.columns)
# print(len(test[["h3_hexagon_high", "time_bucket"]].drop_duplicates()))
print("# of unique hexagons:", test["pickup_h3_res6"].nunique())
# print("# of unique dates:", test["time_bucket"].dt.date.nunique())

359201
Index(['hour_stamp_since_epoch', 'pickup_h3_res6', 'time_bucket',
       'trip_seconds', 'trip_miles', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'pickup_centroid_latitude', 'pickup_centroid_longitude',
       'dropoff_centroid_latitude', 'dropoff_centroid_longitude', 'index',
       'relative_humidity_2m', 'surface_pressure', 'wind_speed_100m',
       'direct_radiation', 'avg_idle_time', 'rush_hour', 'trip_count',
       'active_taxis', 'start_month', 'start_hour', 'day_of_week',
       'is_weekend', 'is_rush_hour', 'is_holiday', 'temperature_2m',
       'apparent_temperature', 'precipitation', 'snowfall', 'snow_depth',
       'wind_speed_10m', 'cloud_cover', 'is_day', 'rain', 'sunshine_duration',
       'start_hour_sin', 'start_hour_cos', 'start_month_sin',
       'start_month_cos', 'day_of_week_sin', 'day_of_week_cos',
       'avg_trip_duration', 'avg_trip_distance', 'avg_fare', 'avg_trip_total',
       'avg_tip', 'tip_rate'],
      dtype='str')
# of unique hexag

In [56]:
test.head()

,hour_stamp_since_epoch,pickup_h3_res6,time_bucket,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total,...,start_month_sin,start_month_cos,day_of_week_sin,day_of_week_cos,avg_trip_duration,avg_trip_distance,avg_fare,avg_trip_total,avg_tip,tip_rate
0,482136,86266452fffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
1,482136,862664c37ffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
2,482136,862664ce7ffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
3,482136,862664197ffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
4,482136,862664557ffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0


In [57]:
print("column: trip_count:", len(test["trip_count"]))
print("column: trip_count NaN values:", sum(test["trip_count"]== 0))

column: trip_count: 359201
column: trip_count NaN values: 350226


In [58]:
# check for missing hours in time_bucket
expected = pd.date_range(start="2025-01-01 00:00:00", 
                         end="2026-01-01 00:00:00", 
                         freq="2h")

missing = expected.difference(test["time_bucket"])

print(f"Missing hours: {len(missing)}")

Missing hours: 0


In [59]:
# datasets["1h_high"]

In [60]:
test.head()

,hour_stamp_since_epoch,pickup_h3_res6,time_bucket,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total,...,start_month_sin,start_month_cos,day_of_week_sin,day_of_week_cos,avg_trip_duration,avg_trip_distance,avg_fare,avg_trip_total,avg_tip,tip_rate
0,482136,86266452fffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
1,482136,862664c37ffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
2,482136,862664ce7ffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
3,482136,862664197ffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
4,482136,862664557ffffff,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.866025,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0,0.0
